In [14]:
from pathlib import Path
from typing import Literal
from dataclasses import dataclass
import math

from IPython.display import display, Markdown
from five_safes_tes_workbench.workbench import Workbench
from monoidal_aggregation.inner_functions import SumOfSquaresIntermediate
from monoidal_aggregation.outer_functions import sum_intermediates, gather_avg_intermediates, gather_var_intermediates
from omop_metadata_utils import DistributionCodesets, count_bar
import pandas as pd
import numpy as np
from scipy import stats

In [2]:
def collect_var_data(path: Path, group_var: str = "condition_status", group_level: str = "with") -> SumOfSquaresIntermediate:
    data = pd.read_csv(path)
    row_data = data[data[group_var] == group_level]
    return SumOfSquaresIntermediate(
        count=row_data["count"].iloc[0],
        sum=row_data["sum"].iloc[0],
        sumsq=row_data["sumsq"].iloc[0]
    )

collect_var_data("data/neoplasm-systol-tre1.csv", group_level="with")

SumOfSquaresIntermediate(sum=np.float64(63987.0), sumsq=np.float64(7640471.0), count=np.int64(536))

In [3]:
class GroupStatsAggregator:
    def __init__(
        self,
        paths: list[Path],
        group_var: str = "condition_status",
        group_levels: list[str] = ["with", "without"]
    ):
        self._paths = paths
        self._group_var = group_var
        self._group_levels = group_levels
        self._data: dict[str, SumOfSquaresIntermediate] = self.collect_data()

    def collect_data(self):
        return {
            gl:[collect_var_data(p, group_var=self._group_var, group_level=gl) for p in self._paths]
            for gl in self._group_levels
        }

    def data(self, key: str) -> list[SumOfSquaresIntermediate]:
        return self._data[key]

    @property
    def count(self) -> dict[str, int]:
        return {
            gl: sum_intermediates.run([v.count for v in value])
            for gl, value in self._data.items()
        }
        
    @property
    def mean(self) -> dict[str, np.float64]:
        return {
            gl: gather_avg_intermediates.run(value)
            for gl, value in self._data.items()
        }

    @property
    def variance(self) -> dict[str, np.float64]:
        return {
            gl: gather_var_intermediates.run(value)
            for gl, value in self._data.items()
        }

    @property
    def std(self) -> dict[str, np.float64]:
        var = self.variance
        return {gl: np.sqrt(value) for gl, value in var.items()}
    

In [4]:
data_paths = [Path("data/neoplasm-systol-tre1.csv"), Path("data/neoplasm-systol-tre2.csv")]

In [5]:
agg = GroupStatsAggregator(data_paths)

In [35]:
@dataclass
class Sample:
    name: str
    data: list[SumOfSquaresIntermediate]

    @property
    def stats(self) -> tuple[int, float, float]:
        return (
            sum_intermediates.run([v.count for v in self.data]),
            gather_avg_intermediates.run(self.data),
            gather_var_intermediates.run(self.data)
        )

class TwoSampleComparator:
    def __init__(
        self,
        sample_1: Sample,
        sample_2: Sample
    ) -> None:
        self._sample_1 = sample_1
        self._sample_2 = sample_2

    def two_sample_t_stat(self, equal_var: bool = False) -> tuple[float, int]:
        n_a, mean_a, var_a = self._sample_1.stats
        n_b, mean_b, var_b = self._sample_2.stats
        
        std_a = np.sqrt(var_a)
        std_b = np.sqrt(var_b)
        return stats.ttest_ind_from_stats(
            mean_a, std_a, n_a,
            mean_b, std_b, n_b,
            equal_var=equal_var
        )

    def cohen_d(self) -> np.float64:
        n_1, x_1, s_1 = self._sample_1.stats
        n_2, x_2, s_2 = self._sample_2.stats
        s = math.sqrt(((n_1 - 1) * s_1 + (n_2 - 1) * s_2) / (n_1 + n_2 - 2))
        return abs(x_1 - x_2) / s

    def report(self, equal_var: bool = False) -> Markdown:
        t_test = self.two_sample_t_stat(equal_var)
        d = self.cohen_d()
        stats_1 = self._sample_1.stats
        stats_2 = self._sample_2.stats
        return Markdown(
            f"""
|   | {self._sample_1.name} | {self._sample_2.name} |
| --- | --- | --- |
| count | {stats_1[0]} | {stats_2[0]} |
| mean | {stats_1[1]:.2f} | {stats_2[1]:.2f} |
| std | {np.sqrt(stats_1[2]):.2f} | {np.sqrt(stats_2[2]):.2f} |

$P$ = {t_test.pvalue:.2f}

Cohen's $d$ = {d:.2f}
            """
        )

In [36]:
compare = TwoSampleComparator(
    Sample(name="Patients with primary neoplasm of skin", data=agg.data("with")),
    Sample(name="Patients without primary neoplasm of skin", data=agg.data("without"))
)

compare.report()


|   | Patients with primary neoplasm of skin | Patients without primary neoplasm of skin |
| --- | --- | --- |
| count | 1039 | 98375 |
| mean | 119.49 | 115.69 |
| std | 1.83 | 6.63 |

$P$ = 0.00

Cohen's $d$ = 0.58
            